# Exp130 — Multilingual BM25 → exact dense document/passage cascade

This is the full 1,200-query **final public-test architecture experiment**, not a score-tuning sweep. It follows the mentor-day example while keeping the official Vietnamese query for dense scoring and source evidence.

**Stages:** cached multilingual BM25 candidates (currently top-500 per VI/EN/ZH branch, up to ~1,500 unique docs/query) → BGE-M3 exact dense scoring over each candidate document's existing chunk vectors → top-400 dense documents → emit top-216 docs → rank exact source chunks within those docs and emit top-96 passages/query. Every submitted chunk is sliced verbatim from the indexed source text using the official chunk offsets.

The mentor slide's `2,000 → 400 → 3,200 → 400` is guidance, not a confirmed BTC baseline or output cap. This experiment uses the BM25 cache that is actually available and records its branch limit. No closed-source API or labels are used.

## Kaggle setup

1. Upload `data/kaggle_inputs/exp130_bm25_dense_passage_input.zip` as a **private** Kaggle Dataset.
2. Attach the **private active index** dataset containing `chunks_metadata.parquet`, `emb_dense/vec_*.npy`, `emb_dense/ids_*.npy`, and `text/part_*.parquet`.
3. Enable a T4 GPU and internet for the first BGE-M3 download. The notebook runs all 1,200 queries and saves a query-disjoint checkpoint every 25 queries; rerunning resumes only when input/index/model/config fingerprints match.
4. Run the notebook. Set `QUERY_LIMIT=120` for a throughput/shape pilot; set it to `None` for the complete 1,200-query submission. Partial runs intentionally do not make a submission ZIP. The exact Hugging Face commit resolved at runtime is saved in the manifest.

Output ZIP is saved in `/kaggle/working/exp130/`. Download and submit manually. The cell fails if schema, source offsets, CRC, or the decimal 100 MB limit fails.

In [ ]:
%pip install -q "FlagEmbedding==1.3.5" "transformers==4.51.3" "accelerate==1.6.0" "sentence-transformers==3.4.1" "huggingface-hub==0.30.2" "pyarrow>=18,<21" "pandas>=2.2,<3

In [ ]:
from __future__ import annotations
import gc, hashlib, json, os, time, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch

INPUT_ROOT = Path('/kaggle/input')
WORK = Path('/kaggle/working/exp130')
WORK.mkdir(parents=True, exist_ok=True)
QUERY_LIMIT = None      # Final full run: all 1,200 official queries.
BM25_DOC_POOL = 1500    # Up to 500 per lexical branch in the existing local cache.
DENSE_DOC_RERANK = 400
OUTPUT_DOCS = 216       # Keep the established output count for a controlled doc-stage test.
OUTPUT_CHUNKS = 96      # Evidence count is fixed for this experiment; source chunks stay verbatim.
RRF_K = 60
BRANCH_WEIGHTS = {'vi': 1.0, 'zh': 1.0, 'en': 0.5}
MODEL_ID = 'BAAI/bge-m3'

zip_paths = list(INPUT_ROOT.rglob('exp130_bm25_dense_passage_input.zip'))
index_roots = [p.parent for p in INPUT_ROOT.rglob('chunks_metadata.parquet')
               if (p.parent/'emb_dense').is_dir() and (p.parent/'text').is_dir()]
if len(zip_paths) != 1:
    raise RuntimeError(f'Expected one private Exp130 input ZIP, found {len(zip_paths)}')
if len(index_roots) != 1:
    raise RuntimeError(f'Expected one attached active index root, found {len(index_roots)}')
INDEX_ROOT = index_roots[0]
INPUT_DIR = WORK/'input'
INPUT_DIR.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_paths[0]) as zf:
    if zf.testzip() is not None: raise RuntimeError('Input ZIP CRC failed')
    zf.extractall(INPUT_DIR)
BM25_PATH = INPUT_DIR/'bm25_candidates.parquet'
QUERY_PATH = INPUT_DIR/'query_translated.parquet'
print({'index_root': str(INDEX_ROOT), 'input_zip': str(zip_paths[0]),
       'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
       'free_working_gb': round(__import__('shutil').disk_usage('/kaggle/working').free/1e9, 1)})

In [ ]:
queries = pd.read_parquet(QUERY_PATH).sort_values('id').reset_index(drop=True)
bm25 = pd.read_parquet(BM25_PATH, columns=['query_id','branch','rank','doc_id','search_version','branch_k'])
if len(queries) != 1200 or queries.id.nunique() != 1200: raise RuntimeError('Expected 1,200 queries')
if set(bm25.search_version) != {'bm25f_idf_pruned_v1'}: raise RuntimeError('Unexpected BM25 cache version')
if set(bm25.branch) != {'vi','en','zh'}: raise RuntimeError('Expected VI/EN/ZH BM25 branches')
if set(bm25.query_id.astype(int)) != set(queries.id.astype(int)): raise RuntimeError('Query ID mismatch')
branch_k = sorted(map(int, bm25.branch_k.unique()))
if branch_k != [500]: raise RuntimeError(f'This plan expects the audited top-500/branch cache, found {branch_k}')

if QUERY_LIMIT is None:
    run_queries = queries.copy()
else:
    if not 1 <= QUERY_LIMIT <= 1200: raise ValueError('QUERY_LIMIT must be None or 1..1200')
    run_queries = queries.iloc[:QUERY_LIMIT].copy()

candidate_docs = {}
for qid, frame in bm25[bm25.query_id.isin(run_queries.id)].groupby('query_id', sort=False):
    scores = {}
    for row in frame.itertuples(index=False):
        scores[int(row.doc_id)] = scores.get(int(row.doc_id), 0.0) + BRANCH_WEIGHTS[str(row.branch)]/(RRF_K+int(row.rank))
    candidate_docs[int(qid)] = [doc for doc,_ in sorted(scores.items(), key=lambda x:(-x[1],x[0]))[:BM25_DOC_POOL]]
if set(candidate_docs) != set(run_queries.id.astype(int)): raise RuntimeError('Candidate coverage incomplete')
print({'queries':len(run_queries), 'bm25_rows':len(bm25), 'branch_k':branch_k,
       'candidate_docs_min':min(map(len,candidate_docs.values())),
       'candidate_docs_mean':round(float(np.mean(list(map(len,candidate_docs.values())))),1),
       'candidate_docs_max':max(map(len,candidate_docs.values()))})

In [ ]:
from FlagEmbedding import BGEM3FlagModel
from huggingface_hub import snapshot_download

if not torch.cuda.is_available(): raise RuntimeError('A Kaggle GPU is required for the full run')
model_path = snapshot_download(repo_id=MODEL_ID)
MODEL_REVISION = Path(model_path).name
encoder = BGEM3FlagModel(model_path, use_fp16=True)
started = time.time()
encoded = encoder.encode(run_queries['query'].astype(str).tolist(), batch_size=8, max_length=1024,
                         return_dense=True, return_sparse=False, return_colbert_vecs=False)
qvec = np.asarray(encoded['dense_vecs'], dtype=np.float32)
qvec /= np.maximum(np.linalg.norm(qvec, axis=1, keepdims=True), 1e-12)
del encoded, encoder
gc.collect(); torch.cuda.empty_cache()
print({'query_vectors':qvec.shape, 'encode_seconds':round(time.time()-started,1),
       'gpu_memory_gb':round(torch.cuda.max_memory_allocated()/1e9,2)})

In [ ]:
# Validate the active vector shards and construct the doc→chunk map from the official metadata.
meta = pd.read_parquet(INDEX_ROOT/'chunks_metadata.parquet', columns=['chunk_id','doc_id','start','end'])
meta = meta.sort_values('chunk_id').reset_index(drop=True)
if not np.array_equal(meta.chunk_id.to_numpy(), np.arange(len(meta), dtype=np.int64)):
    raise RuntimeError('Active chunk IDs are not contiguous/sorted as required')
doc_chunks = meta.groupby('doc_id', sort=False).chunk_id.apply(lambda s:s.to_numpy(dtype=np.int32)).to_dict()
doc_arr = meta.doc_id.to_numpy(dtype=np.int64)
start_arr = meta.start.to_numpy(dtype=np.int64)
end_arr = meta.end.to_numpy(dtype=np.int64)

id_paths = sorted((INDEX_ROOT/'emb_dense').glob('ids_*.npy'))
vec_paths = sorted((INDEX_ROOT/'emb_dense').glob('vec_*.npy'))
if len(id_paths) != len(vec_paths) or not id_paths: raise RuntimeError('Dense shard set is incomplete')
vec_gpu, bounds = [], []
offset = 0
for id_path, vec_path in zip(id_paths, vec_paths):
    ids = np.load(id_path, mmap_mode='r')
    vec = np.load(vec_path, mmap_mode='r')
    if vec.shape != (len(ids),1024) or int(ids[0]) != offset or not np.array_equal(ids, np.arange(offset,offset+len(ids),dtype=np.int64)):
        raise RuntimeError(f'Chunk/vector shard alignment failed: {id_path.name}')
    vec_gpu.append(torch.from_numpy(np.array(vec, copy=True)).to('cuda', non_blocking=True))
    bounds.append((offset, offset+len(ids)))
    offset += len(ids)
if offset != len(meta): raise RuntimeError(f'Vector rows {offset:,} != metadata rows {len(meta):,}')
print({'chunks':len(meta), 'docs':len(doc_chunks), 'dense_shards':len(vec_gpu),
       'dense_gb_gpu':round(sum(x.numel()*x.element_size() for x in vec_gpu)/1e9,2),
       'gpu_allocated_gb':round(torch.cuda.memory_allocated()/1e9,2)})

In [ ]:
# Exact BGE-M3 scoring on the BM25 candidate docs' existing chunk vectors.
def fetch_dense_scores(chunk_ids: np.ndarray, query_vector: np.ndarray) -> np.ndarray:
    scores = np.empty(len(chunk_ids), dtype=np.float32)
    for part, (lo, hi) in enumerate(bounds):
        mask = (chunk_ids >= lo) & (chunk_ids < hi)
        if not mask.any(): continue
        local = torch.as_tensor(chunk_ids[mask].astype(np.int64)-lo, device='cuda')
        vectors = vec_gpu[part].index_select(0, local).float()
        q = torch.as_tensor(query_vector, device='cuda', dtype=torch.float32)
        scores[mask] = (vectors @ q).detach().cpu().numpy()
    if not np.isfinite(scores).all(): raise RuntimeError('Non-finite dense scores')
    return scores

rankings = {}
passages = {}
checkpoint_npz = WORK/'exp130_checkpoint.npz'
fingerprint = {
    'input_zip_sha256': hashlib.sha256(zip_paths[0].read_bytes()).hexdigest(),
    'index_manifest_sha256': hashlib.sha256((INDEX_ROOT/'rebuild_manifest.json').read_bytes()).hexdigest()
        if (INDEX_ROOT/'rebuild_manifest.json').exists() else None,
    'model_id': MODEL_ID, 'model_revision': MODEL_REVISION,
    'query_ids_sha256': hashlib.sha256(np.asarray(run_queries.id,dtype=np.int64).tobytes()).hexdigest(),
    'bm25_doc_pool': BM25_DOC_POOL, 'dense_doc_rerank': DENSE_DOC_RERANK,
    'output_docs': OUTPUT_DOCS, 'output_chunks': OUTPUT_CHUNKS,
    'rrf_k': RRF_K, 'branch_weights': BRANCH_WEIGHTS,
}
if checkpoint_npz.exists():
    with np.load(checkpoint_npz, allow_pickle=False) as saved_arrays:
        saved_fingerprint=json.loads(str(saved_arrays['fingerprint_json'].item()))
        done_ids = saved_arrays['query_ids'].astype(np.int64)
        saved_docs = saved_arrays['docs'].astype(np.int64)
        saved_passage_ids = saved_arrays['passage_ids'].astype(np.int32)
        saved_scores = saved_arrays['passage_scores'].astype(np.float32)
        completed_queries=int(saved_arrays['completed_queries'].item())
    if saved_fingerprint != fingerprint:
        raise RuntimeError('Checkpoint fingerprint differs from current data/model/config')
    if completed_queries != len(done_ids):
        raise RuntimeError('Checkpoint completion counter is inconsistent')
    expected_ids = run_queries.id.to_numpy(dtype=np.int64)[:len(done_ids)]
    if (not np.array_equal(done_ids, expected_ids) or saved_docs.shape != (len(done_ids),OUTPUT_DOCS)
            or saved_passage_ids.shape != (len(done_ids),OUTPUT_CHUNKS)
            or saved_scores.shape != (len(done_ids),OUTPUT_CHUNKS)):
        raise RuntimeError('Checkpoint query order or array shapes are invalid')
    for i,qid in enumerate(done_ids):
        rankings[int(qid)] = saved_docs[i].tolist()
        passages[int(qid)] = [(int(cid),int(doc_arr[int(cid)]),float(score))
                               for cid,score in zip(saved_passage_ids[i],saved_scores[i],strict=True)]
    print({'resumed_queries':len(done_ids),'remaining_queries':len(run_queries)-len(done_ids)},flush=True)

def save_checkpoint():
    ids=np.asarray(list(rankings),dtype=np.int64)
    doc_array=np.asarray(list(rankings.values()),dtype=np.int64)
    passage_ids=np.asarray([[x[0] for x in passages[q]] for q in rankings],dtype=np.int32)
    passage_scores=np.asarray([[x[2] for x in passages[q]] for q in rankings],dtype=np.float32)
    temp_npz=checkpoint_npz.with_suffix('.tmp.npz')
    np.savez_compressed(temp_npz,query_ids=ids,docs=doc_array,
                        passage_ids=passage_ids,passage_scores=passage_scores,
                        fingerprint_json=np.asarray(json.dumps(fingerprint,sort_keys=True)),
                        completed_queries=np.asarray(len(ids),dtype=np.int64))
    os.replace(temp_npz,checkpoint_npz)

started = time.time()
start_index=len(rankings)
for qi in range(start_index,len(run_queries)):
    row = next(run_queries.iloc[[qi]].itertuples(index=False))
    qid = int(row.id)
    docs = [d for d in candidate_docs[qid] if d in doc_chunks]
    if len(docs) < OUTPUT_DOCS: raise RuntimeError(f'Only {len(docs)} valid docs for query {qid}')
    lengths = np.fromiter((len(doc_chunks[d]) for d in docs), dtype=np.int32, count=len(docs))
    chunk_ids = np.concatenate([doc_chunks[d] for d in docs]).astype(np.int32, copy=False)
    chunk_doc_index = np.repeat(np.arange(len(docs), dtype=np.int32), lengths)
    chunk_scores = fetch_dense_scores(chunk_ids, qvec[qi])
    doc_scores = np.full(len(docs), -np.inf, dtype=np.float32)
    np.maximum.at(doc_scores, chunk_doc_index, chunk_scores)
    doc_order = np.argsort(-doc_scores, kind='stable')[:DENSE_DOC_RERANK]
    top_docs = [docs[i] for i in doc_order]
    final_docs = top_docs[:OUTPUT_DOCS]
    keep_doc_set = set(final_docs)
    keep_mask = np.fromiter((docs[i] in keep_doc_set for i in chunk_doc_index), dtype=bool, count=len(chunk_ids))
    eligible = np.flatnonzero(keep_mask)
    passage_order = eligible[np.argsort(-chunk_scores[eligible], kind='stable')[:OUTPUT_CHUNKS]]
    if len(passage_order) != OUTPUT_CHUNKS: raise RuntimeError(f'Insufficient chunk candidates for {qid}')
    rankings[qid] = final_docs
    passages[qid] = [(int(chunk_ids[i]), int(doc_arr[int(chunk_ids[i])]), float(chunk_scores[i])) for i in passage_order]
    completed_this_run=qi-start_index+1
    if completed_this_run%25 == 0 or qi+1 == len(run_queries):
        save_checkpoint()
        elapsed = time.time()-started
        remaining=len(run_queries)-(qi+1)
        print(f'{qi+1}/{len(run_queries)} queries | {completed_this_run/max(elapsed,1e-6):.2f} q/s | ETA {remaining*elapsed/max(completed_this_run,1)/60:.1f} min | checkpoint saved', flush=True)

if len(rankings) != 1200:
    raise RuntimeError(f'Expected all 1,200 query rankings, found {len(rankings)}')
np.savez_compressed(WORK/'exp130_rankings.npz', query_ids=np.asarray(list(rankings),dtype=np.int64),
                    docs=np.asarray(list(rankings.values()),dtype=np.int64),
                    passage_ids=np.asarray([[x[0] for x in passages[q]] for q in rankings],dtype=np.int32),
                    passage_scores=np.asarray([[x[2] for x in passages[q]] for q in rankings],dtype=np.float32))
print({'docs_per_query':OUTPUT_DOCS,'dense_doc_rerank_k':DENSE_DOC_RERANK,
       'passages_per_query':OUTPUT_CHUNKS,'queries_done':len(rankings)})

In [ ]:
if QUERY_LIMIT is not None and len(run_queries) != 1200:
    raise RuntimeError('Pilot only: set QUERY_LIMIT=None and rerun before creating a submission')

# Load original source text only for documents that own a selected chunk.
needed_docs = {doc_id for qid in rankings for _cid,doc_id,_score in passages[qid]}
text_by_doc = {}
text_paths = sorted((INDEX_ROOT/'text').glob('part_*.parquet'))
started = time.time()
for i,path in enumerate(text_paths,1):
    frame = pd.read_parquet(path, columns=['doc_id','text'])
    sub = frame[frame.doc_id.isin(needed_docs-set(text_by_doc))]
    text_by_doc.update((int(r.doc_id),str(r.text)) for r in sub.itertuples(index=False))
    if i%10 == 0 or i == len(text_paths):
        print(f'text shards {i}/{len(text_paths)} | recovered {len(text_by_doc):,}/{len(needed_docs):,}', flush=True)
missing = needed_docs-set(text_by_doc)
if missing: raise RuntimeError(f'Missing {len(missing):,} source docs for evidence')

submission=[]
for qid in rankings:
    chunks=[]
    selected_docs=set(rankings[qid])
    for chunk_id,doc_id,_score in passages[qid]:
        if doc_id not in selected_docs: raise RuntimeError('Chunk/doc hierarchy broken')
        chunk_text=text_by_doc[doc_id][int(start_arr[chunk_id]):int(end_arr[chunk_id])]
        if not chunk_text.strip(): raise RuntimeError(f'Empty source slice {qid}/{chunk_id}')
        chunks.append({'doc_id':int(doc_id),'chunk_text':chunk_text})
    if len(set(rankings[qid])) != OUTPUT_DOCS or len(chunks) != OUTPUT_CHUNKS:
        raise RuntimeError(f'Wrong output cardinality for query {qid}')
    submission.append({'id':int(qid),'relevant_docs':list(map(int,rankings[qid])),'relevant_chunks':chunks})
submission.sort(key=lambda x:x['id'])
json_path=WORK/'submission_exp130_bm25_dense_passage.json'
zip_path=WORK/'submission_exp130_bm25_dense_passage.zip'
json_path.write_text(json.dumps(submission,ensure_ascii=False,separators=(',',':')),encoding='utf-8')
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_LZMA,compresslevel=9) as zf:
    zf.write(json_path,'submission.json')
with zipfile.ZipFile(zip_path) as zf:
    assert zf.namelist()==['submission.json'] and zf.testzip() is None
if zip_path.stat().st_size >= 100_000_000:
    raise RuntimeError(f'ZIP is over decimal 100 MB: {zip_path.stat().st_size:,}')
def file_sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''): h.update(block)
    return h.hexdigest()
manifest={
    'experiment':130,'model':MODEL_ID,'model_revision':MODEL_REVISION,
    'bm25_search_version':'bm25f_idf_pruned_v1','bm25_branch_k':branch_k,
    'bm25_candidate_pool_per_query':BM25_DOC_POOL,'rrf_k':RRF_K,
    'dense_doc_rerank_k':DENSE_DOC_RERANK,'output_docs':OUTPUT_DOCS,
    'output_chunks':OUTPUT_CHUNKS,'queries':len(submission),
    'query_ids':[int(x['id']) for x in submission],
    'input_zip_sha256':file_sha(zip_paths[0]),
    'index_manifest_sha256':file_sha(INDEX_ROOT/'rebuild_manifest.json') if (INDEX_ROOT/'rebuild_manifest.json').exists() else None,
    'json_sha256':file_sha(json_path),'zip_sha256':file_sha(zip_path),
    'zip_bytes':zip_path.stat().st_size,'zip_lzma_crc_pass':True,
}
(WORK/'exp130_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print({**manifest,'path':str(zip_path),'manifest_path':str(WORK/'exp130_manifest.json')})

## Interpretation and next Kaggle pass

This run changes the document candidate/scoring architecture and passage selection together; it is a **large ablation**, not a claim that a score gain is guaranteed. If the official score changes, attribute axes using Docs F2/Chunks F2 and keep the JSON/manifest. If time remains, a follow-up should freeze the winning doc list and change only passage selection, then benchmark a legal open reranker on a 20–40 query pilot before scoring all 1,200.

Do not upload the index or input dataset publicly. Preserve the output ZIP, `exp130_rankings.npz`, input manifest and Kaggle run settings for reproducibility.